# IMDB Review Analysis

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import nltk   # nltk - Natural Language Tool Kit
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

# To speed up the process
from joblib import Parallel, delayed
from multiprocessing import cpu_count

# Import Regular Expression
import re

# Warnings
import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/seethamounika/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Load the Data

In [2]:
data = pd.read_csv('/Users/seethamounika/Desktop/DL/DL(Docs)/imdb.csv')

## Initial Analysis

In [3]:
data.head()

,text,label
0,In Panic In The Streets Richard Widmark plays ...,1
1,If you ask me the first one was really better ...,0
2,I am a big fan a Faerie Tale Theatre and I've ...,1
3,I just finished reading a book about Dillinger...,0
4,Greg Davis and Bryan Daly take some crazed sta...,0


In [4]:
data.tail()

,text,label
24995,My roommates & I nearly shorted out our TV fro...,0
24996,Michelle Rodriguez is the defining actress who...,1
24997,Nice movie with a great soundtrack which spans...,1
24998,"Even though this was a made-for-TV production,...",0
24999,I saw this on cable recently and kinda enjoyed...,0


In [5]:
data.shape

(25000, 2)

In [6]:
data['label'].value_counts()

label
1    12500
0    12500
Name: count, dtype: int64

## Preprocess

In [7]:
ps = PorterStemmer()
STOPWORDS = set(stopwords.words('english'))

In [8]:
def preprocess(x):
    x = re.sub('[^a-zA-Z ]', " ", x)
    x = x.lower()
    x = x.split()
    x = [word for word in x if word not in STOPWORDS]
    x = [ps.stem(word) for word in x]
    x = " ".join(x)
    return x

data['text'] = data['text'].apply(preprocess)
data.head()

,text,label
0,panic street richard widmark play u navi docto...,1
1,ask first one realli better one look sarah g r...,0
2,big fan faeri tale theatr seen one best funni ...,1
3,finish read book dilling movi horribl inaccur ...,0
4,greg davi bryan dali take craze statement terr...,0


In [9]:
data.text.values

array(['panic street richard widmark play u navi doctor week rude interrupt corps contain plagu cop paul dougla properli point guy die two bullet chest issu two becom unwil partner effort find killer anyon els expos diseas br br point number peopl reason director elia kazan bother cast small part anyon sound like louisiana new orlean stori take place person attest richard widmark wife barbara bel gedd excus navi doctor could assign nativ work br br plagu news kept secret new orlean pd start dragnet citi underworld dead guy came ship europ underworld connect new orlean wise guy play jack palanc jump whole bunch erron conclus start harass cousin dead guy start show plagu symptom palanc got rave review first film receiv notic br br person favorit film zero mostel happen right mostel blacklist around time made specialti play would tough guy realli toadi play kind role humphrey bogart film enforc sadli kind identifi mostel last chase scene palanc chase widmark dougla half new orlean polic s

In [10]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [11]:
a=np.array(["the sun is shining",
            "the weather is good ",
            "the sun is shining and the weather is good"])

cv = CountVectorizer()
a = cv.fit_transform(a).toarray()

a = pd.DataFrame(a, columns=cv.get_feature_names_out())
a

,and,good,is,shining,sun,the,weather
0,0,0,1,1,1,1,0
1,0,1,1,0,0,1,1
2,1,1,2,1,1,2,1


In [12]:
x = cv.fit_transform(data['text'].values).toarray()
x = pd.DataFrame(x, columns = cv.get_feature_names_out())
x.sample(5)

,aa,aaa,aaaaaaah,aaaaah,aaaaatch,aaaahhhhhhh,aaaand,aaaarrgh,aaah,aaargh,...,zyurang,zz,zzzz,zzzzz,zzzzzzzz,zzzzzzzzzzzz,zzzzzzzzzzzzz,zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz,zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz,zzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzzz
8739,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8686,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
19580,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3641,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9632,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
x.shape

(25000, 49642)

## Splitting the Data

In [14]:
from sklearn.model_selection import train_test_split

X = data['text'].values
y = data['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [15]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 20000     # adjust if needed
MAX_LEN = 200          # max words per review

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post")

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 55s 216ms/step - accuracy: 0.5055 - loss: 0.6932 - val_accuracy: 0.5235 - val_loss: 0.6891
Epoch 2/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 59s 234ms/step - accuracy: 0.5446 - loss: 0.6753 - val_accuracy: 0.5362 - val_loss: 0.6665
Epoch 3/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 60s 241ms/step - accuracy: 0.5625 - loss: 0.6362 - val_accuracy: 0.5422 - val_loss: 0.6744
Epoch 4/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 64s 255ms/step - accuracy: 0.6320 - loss: 0.5924 - val_accuracy: 0.7023 - val_loss: 0.6166
Epoch 5/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 66s 264ms/step - accuracy: 0.6888 - loss: 0.5837 - val_accuracy: 0.5965 - val_loss: 0.6637
Epoch 6/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 66s 265ms/step - accuracy: 0.6484 - loss: 0.5772 - val_accuracy: 0.7735 - val_loss: 0.5978
Epoch 7/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 69s 278ms/step - accuracy: 0.8231 - loss: 0.4137 - val_accuracy: 0.8390 - val_loss: 0.4117
Epoch 8/30
250/250 ━━━━━━━━━━━━━━━━━━━━ 65s 260ms/step - accuracy: 0.8949 - loss: 0

In [18]:
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", accuracy)

157/157 ━━━━━━━━━━━━━━━━━━━━ 12s 75ms/step - accuracy: 0.8554 - loss: 1.0143
Test Accuracy: 0.855400025844574


In [19]:
def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
    prob = model.predict(pad)[0][0]
    return "Positive" if prob >= 0.5 else "Negative"

# Example
predict_sentiment("Good movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step


'Negative'

In [20]:
predict_sentiment('This is worst movie, I watched ever')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


'Negative'

## Streamlit app

In [22]:
# 4. SAVE MODEL
import pickle
model.save("sentiment_model.keras")


# 5. SAVE TOKENIZER (PICKLE)
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Model and tokenizer saved successfully")

✅ Model and tokenizer saved successfully


In [23]:
# predict.py

import pickle
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# LOAD MODEL
model = load_model("sentiment_model.keras")


# LOAD TOKENIZER (PICKLE)
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

MAX_LEN = 200


# PREDICTION FUNCTION
def predict_sentiment(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post")
    prob = model.predict(pad, verbose=0)[0][0]
    return "Positive" if prob >= 0.5 else "Negative"

# EXAMPLE
print(predict_sentiment("Good movie"))
print(predict_sentiment("Worst film ever"))

Negative
Negative


In [24]:
predict_sentiment('Very good movie in recent times')

'Positive'

In [25]:
predict_sentiment('Good move')

'Negative'